## 1 · Upload and extract


In [ ]:
from pathlib import Path
import zipfile
BASE = Path('/data/epic20/APGCE/apgce_team_13')
BASE.mkdir(parents=True, exist_ok=True)
archives = [BASE/'ANGSI_JR_V8_2_FULL.zip', Path.cwd()/'ANGSI_JR_V8_2_FULL.zip', BASE/'ANGSI_V82_CODE.zip', Path.cwd()/'ANGSI_V82_CODE.zip']
archive = next((p for p in archives if p.exists()), None)
if archive is None:
    raise FileNotFoundError(f'Upload ANGSI_JR_V8_2_FULL.zip into {BASE} or {Path.cwd()}')
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, 'ZIP integrity failed'
    z.extractall(BASE)
APP_DIR = BASE/'ANGSI_JR_V8_2_FULL'
APP = APP_DIR/'angsi_jr_v8.py'
assert APP.exists() and (APP_DIR/'opportunity_intelligence.py').exists()
print('READY:', APP)


## 2 · Install Python packages into this kernel


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(APP_DIR/'requirements.txt')])


## 3 · Launch on port 8511


In [ ]:
import os, sys, py_compile, subprocess
py_compile.compile(str(APP), doraise=True)
env = os.environ.copy()
env['ANGSI_DATA_ROOT'] = '/data/epic20/APGCE/input_data/challenge3'
assert Path(env['ANGSI_DATA_ROOT']).is_dir(), 'Challenge 3 dataset not found'
log_path = APP_DIR/'server_v82.log'
with log_path.open('a') as logfile:
    process_v82 = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', str(APP),
             '--server.address=127.0.0.1', '--server.port=8511', '--server.headless=true',
             '--server.fileWatcherType=none'],cwd=str(APP_DIR),env=env,
             stdout=logfile,stderr=subprocess.STDOUT,start_new_session=True)
print('PID:',process_v82.pid,'URL on HPC: http://127.0.0.1:8511')


## 4 · Verify health


In [ ]:
import time, urllib.request
for i in range(15):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8511/_stcore/health',timeout=3) as reply:
            print('SERVER HEALTH:', reply.status, reply.read().decode()); break
    except Exception:
        time.sleep(1)
else:
    print(log_path.read_text(errors='replace')[-6000:])


## 5 · Optional unit tests (requires pytest)


In [ ]:
import subprocess,sys
result=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_v82_opportunity_intelligence.py'],cwd=str(APP_DIR),capture_output=True,text=True)
print(result.stdout or result.stderr)
